# Notebook 1 — Data Clean and Merge

### Input : 

This notebook takes raw data from following CSV files.

      1. Raw_Data/engagements_raw.csv - Engagement fees, clients, project dates, status, and engagement ID
      
      2. Raw_Data/timesheets_raw.csv - Who worked, hours logged, week ending, and engagement ID
      
      3. Raw_Data/consultants_raw.csv  - Consultant names, rates, levels, practices, and hire dates
      

### Output: 

clean_merged.csv - A single clean, merged analysis-ready dataset which can be passed to Notebook 2 for Data Visualization and Report Generation.

In [1]:
# install pandas library
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: /Library/Frameworks/Python.framework/Versions/3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## A. Data Cleaning : engagements_raw file

In [2]:
import os
import pandas as pd

# Define the folder and file name
folder_path = "Raw_Data"
file_name = "engagements_raw.csv"

# Join folder and file name into
file_path = os.path.join(folder_path, file_name)

print("File to be loaded -->", file_path)
print("-" * 50)
try:
    # Check if the parent folder exists and raise error if the Raw_Data folder is missing 
    if not os.path.isdir(os.path.dirname(file_path) or "."):
        raise FileNotFoundError(f"Folder not found: {os.path.dirname(file_path)}")
    # Check if the specific file exists inside that folder and raise error if the Raw_Data/engagements_raw.csv file is missing 
    if not os.path.isfile(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    # If both checks pass, load the CSV into a engagements_df DataFrame
    engagements_df = pd.read_csv(file_path)

# Catch the two custom FileNotFoundError cases raised above (missing folder or file)
except FileNotFoundError as e:
    print(f"Error: {e}")
    
# Catch any other unexpected error during the read
except Exception as e:
        print(f"Unexpected error while reading '{file_path}': {e}")

if engagements_df is not None:
    # Print shape, every column data type, numeric statistics, top 7 rows of data
    print("\n")
    print(f"{file_path} loaded successfully!")
    print()
    print(f"Size of engagements DataFrame: {engagements_df.shape[0]} rows, {engagements_df.shape[1]} columns \n")
    print()
    print(f"dtypes : ")
    print(engagements_df.dtypes)
    print()
    print(f"Numeric statistics : ")
    display(engagements_df.describe())
    print()
    display(engagements_df.head(7))

File to be loaded --> Raw_Data/engagements_raw.csv
--------------------------------------------------


Raw_Data/engagements_raw.csv loaded successfully!

Size of engagements DataFrame: 90 rows, 9 columns 


dtypes : 
engagement_id        object
client_name          object
practice             object
lead_consultant      object
start_date           object
end_date             object
project_fee          object
status               object
secondary_contact    object
dtype: object

Numeric statistics : 


,engagement_id,client_name,practice,lead_consultant,start_date,end_date,project_fee,status,secondary_contact
count,90,90,76,90,90,90,90,90,43
unique,86,10,4,8,82,85,19,4,11
top,ENG-043,BrightPath Health,People & Change,Marcus Webb,2023-02-24,2025-11-11,"$230,000",On Hold,Derek Fowler
freq,2,20,23,14,2,2,13,27,7


,engagement_id,client_name,practice,lead_consultant,start_date,end_date,project_fee,status,secondary_contact
0,ENG-001,Apex Financial,Digital,Jordan Reeves,"November 24, 2023","July 2, 2023","$45,000",Active,NaN
1,ENG-002,Orion Energy,People & Change,Anika Patel,2025-05-26,03/23/2026,"$68,000",On Hold,NaN
2,ENG-003,Vanguard Logistics,Strategy,Tomas Herrera,06/20/2024,2025-12-15,"$230,000",Active,Hannah Firth
3,ENG-004,Apex Financial,Operations,Anika Patel,"January 22, 2023",02/28/2024,"$150,000",Active,NaN
4,ENG-005,Titan Manufacturing,Operations,Rachel Kim,05/23/2025,"October 21, 2026","$110,000",Completed,NaN
5,ENG-006,Titan Manufacturing,Digital,Rachel Kim,01/26/2023,2023-02-07,"$78,000",On Hold,NaN
6,ENG-007,Titan Manufacturing,People & Change,Jordan Reeves,2023-04-24,"December 19, 2023","$68,000",Cancelled,NaN


### A1. Using duplicated() function to scan dataFrame row-by-row to find repeating rows in engagements

In [3]:
# Get the total number of duplicate rows
print("Total count of duplicate rows  in engagements : ",engagements_df.duplicated().sum())

# Display all the duplicate rows including the originals 
# (keep=False: Marks all copies as True. This is useful when we want to find every single row involved in a duplication issue.)
duplicate_rows = engagements_df[engagements_df.duplicated(keep=False)] 
duplicate_rows

Total count of duplicate rows  in engagements :  4


,engagement_id,client_name,practice,lead_consultant,start_date,end_date,project_fee,status,secondary_contact
5,ENG-006,Titan Manufacturing,Digital,Rachel Kim,01/26/2023,2023-02-07,"$78,000",On Hold,NaN
12,ENG-006,Titan Manufacturing,Digital,Rachel Kim,01/26/2023,2023-02-07,"$78,000",On Hold,NaN
20,ENG-020,BrightPath Health,Operations,Lena Kowalski,2023-01-15,"February 3, 2024","$230,000",Completed,NaN
35,ENG-020,BrightPath Health,Operations,Lena Kowalski,2023-01-15,"February 3, 2024","$230,000",Completed,NaN
44,ENG-043,Vanguard Logistics,Strategy,Priya Nambiar,2025-07-22,2025-11-11,"$45,000",Cancelled,Derek Fowler
58,ENG-043,Vanguard Logistics,Strategy,Priya Nambiar,2025-07-22,2025-11-11,"$45,000",Cancelled,Derek Fowler
64,ENG-062,BrightPath Health,Operations,Lena Kowalski,09/16/2024,2024-09-13,"$140,000",On Hold,Leo Marchand
78,ENG-062,BrightPath Health,Operations,Lena Kowalski,09/16/2024,2024-09-13,"$140,000",On Hold,Leo Marchand


Rows with 'engagement_id' : ENG-006, ENG-020, ENG-043, ENG-062 have been repeated twice.

### Using drop_duplicates() method to remove duplicate rows from DataFrame

Since we know number of duplicates = 4 & total rows = 90, we expect 90-4 = 86 rows after cleaning.

In [4]:
print(f"Number of rows before removing duplictes in engagements: {len(engagements_df)}")
engagements_df = engagements_df.drop_duplicates()
print(f"Number of rows after removing duplictes in engagements:  {len(engagements_df)}")

# Get the total number of duplicate rows - should return 0 now
print("Total count of duplicate rows in engagements :",engagements_df.duplicated().sum())

Number of rows before removing duplictes in engagements: 90
Number of rows after removing duplictes in engagements:  86
Total count of duplicate rows in engagements : 0


### A2. Using info() to verify column names, data types, non-null counts 

In [5]:
engagements_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 86 entries, 0 to 89
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   engagement_id      86 non-null     object
 1   client_name        86 non-null     object
 2   practice           72 non-null     object
 3   lead_consultant    86 non-null     object
 4   start_date         86 non-null     object
 5   end_date           86 non-null     object
 6   project_fee        86 non-null     object
 7   status             86 non-null     object
 8   secondary_contact  41 non-null     object
dtypes: object(9)
memory usage: 6.7+ KB


If a column reads 'N' non-null out of 'total', we instantly know that column has 'total - N' missing or blank cells (NaN).

'practice' has 72 non-null out of 86 ---> missing values : 86-72 = 14 (few empty)

'secondary_contact' has 41 non-null out of 86 ---> missing values : 86-41 = 45 (50% empty)

### Using isna().sum() to fetch the exact count of missing values in each column of DataFrame.

In [6]:
# To directly check how many missing values we have per column
engagements_df.isna().sum()

engagement_id         0
client_name           0
practice             14
lead_consultant       0
start_date            0
end_date              0
project_fee           0
status                0
secondary_contact    45
dtype: int64

We see that column - 'practice'  has 14 & 'secondary_contact' has 47 missing values which is consistent with our previous calculation with info().

### isna().any(axis=1) finds rows with missing data (NaNs).

any(axis=1) tells Pandas to scan horizontally across each row and select it if at least one column contains an empty or missing cell.

In [7]:
engagements_missing_rows = engagements_df[engagements_df.isna().any(axis=1)]
engagements_missing_rows.head(10)

,engagement_id,client_name,practice,lead_consultant,start_date,end_date,project_fee,status,secondary_contact
0,ENG-001,Apex Financial,Digital,Jordan Reeves,"November 24, 2023","July 2, 2023","$45,000",Active,NaN
1,ENG-002,Orion Energy,People & Change,Anika Patel,2025-05-26,03/23/2026,"$68,000",On Hold,NaN
3,ENG-004,Apex Financial,Operations,Anika Patel,"January 22, 2023",02/28/2024,"$150,000",Active,NaN
4,ENG-005,Titan Manufacturing,Operations,Rachel Kim,05/23/2025,"October 21, 2026","$110,000",Completed,NaN
5,ENG-006,Titan Manufacturing,Digital,Rachel Kim,01/26/2023,2023-02-07,"$78,000",On Hold,NaN
6,ENG-007,Titan Manufacturing,People & Change,Jordan Reeves,2023-04-24,"December 19, 2023","$68,000",Cancelled,NaN
7,ENG-008,Vanguard Logistics,NaN,Priya Nambiar,11/06/2023,"February 13, 2023","$55,000",Cancelled,NaN
8,ENG-009,Orion Energy,NaN,Marcus Webb,"May 25, 2025","May 14, 2025","$110,000",Cancelled,NaN
9,ENG-010,BrightPath Health,Strategy,David Chen,"October 7, 2025",03/18/2025,"$192,000",Active,NaN
10,ENG-011,Titan Manufacturing,Operations,Anika Patel,10/03/2023,02/25/2024,"$230,000",Completed,NaN


In [8]:
# Get the total number of rows with missing values
print(f'There are total {engagements_missing_rows.shape[0]} rows with missing values in engagements.')

There are total 51 rows with missing values in engagements.


### Solution to NULL values :
In our case since both 'practice' and 'secondary_contact'columns contain text information, the best approach is to fill the missing blanks with clear placeholder strings like "Unknown" or "Not assigned".

Using dropna() would delete up to 59 whole rows from your dataset. Since critical columns like engagement_id, client_name, and project_fee are 100% complete, throwing away those rows means we would lose valid revenue data and project records.

#### fillna() to fill missing values of 'practice' column with "Unknown" and 'secondary_contact' with "Not assigned" :

In [9]:
engagements_df["practice"] = engagements_df["practice"].fillna("Unknown")
engagements_df["secondary_contact"] = engagements_df["secondary_contact"].fillna("Not assigned")

# isna().sum() to verify if the number for filled column has dropped to 0.
engagements_df.isna().sum()

engagement_id        0
client_name          0
practice             0
lead_consultant      0
start_date           0
end_date             0
project_fee          0
status               0
secondary_contact    0
dtype: int64

This confirms your fillna() worked perfectly. 14 rows are no longer missing (NaN), and filled as 'Unknown'.

### A3. Apply value_counts() on all string type columns to find mismatched IDs, inconsistent naming, casing or whitespace differences

In [10]:
print(engagements_df['practice'].value_counts())
print("-" * 30)
print(engagements_df['client_name'].value_counts())

practice
People & Change    23
Digital            17
Strategy           16
Operations         16
Unknown            14
Name: count, dtype: int64
------------------------------
client_name
BrightPath Health        18
Titan Manufacturing      15
Orion Energy             13
Apex Financial           12
Vanguard Logistics       11
Vanguard Logistics        5
BrightPath Health         4
Apex Financial            3
Titan Manufacturing       3
Orion Energy              2
Name: count, dtype: int64


'practice' value counts looks clean with missing values are perfectly grouped as 'Unknown'.Whereas there are two different string values in 'client_name' that look identical when printed, but aren't identical to Pandas due to trailing/leading whitespace.

### Apply str.strip() on all string type columns to remove leading/trailing whitespaces 

In [11]:
engagements_df["engagement_id"] = engagements_df["engagement_id"].str.strip()
engagements_df["client_name"] = engagements_df["client_name"].str.strip()
engagements_df["practice"] = engagements_df["practice"].str.strip()
engagements_df["lead_consultant"] = engagements_df["lead_consultant"].str.strip()
engagements_df["secondary_contact"] = engagements_df["secondary_contact"].str.strip()

# Verify no duplicates rows after trimming whitespaces.
print("Total count of duplicate rows in engagements :",engagements_df.duplicated().sum())

Total count of duplicate rows in engagements : 0


##  A4. dtypes to identify Data types of all columns :

In [12]:
engagements_df.dtypes

engagement_id        object
client_name          object
practice             object
lead_consultant      object
start_date           object
end_date             object
project_fee          object
status               object
secondary_contact    object
dtype: object

'project_fee' is object (string)              ----> Convert into valid number format.

'start_date' and 'end_date' are object (string)  ---> Convert into valid date format.


In [13]:
# Print three different date formats in 'start_date' column with type:string object
print("Date formats start_date:")
print(engagements_df["start_date"].head(5))
print("-" * 25)
print("Date formats end_date:")
print(engagements_df["end_date"].head(5))
print("-" * 25)
# Visullay checking 'project_fee' column has $ sign and comma separators and type:string object
print("Project fee format:")
print(engagements_df["project_fee"].head(3))
print("-" * 25)
print(engagements_df[["start_date", "end_date"]].dtypes)

Date formats start_date:
0    November 24, 2023
1           2025-05-26
2           06/20/2024
3     January 22, 2023
4           05/23/2025
Name: start_date, dtype: object
-------------------------
Date formats end_date:
0        July 2, 2023
1          03/23/2026
2          2025-12-15
3          02/28/2024
4    October 21, 2026
Name: end_date, dtype: object
-------------------------
Project fee format:
0     $45,000
1     $68,000
2    $230,000
Name: project_fee, dtype: object
-------------------------
start_date    object
end_date      object
dtype: object


## to_datetime() to convert the 'start_date' and 'end_date' columns to valid datetime64 objects.

In [14]:
engagements_df["start_date"] = pd.to_datetime(engagements_df["start_date"], format="mixed")
engagements_df["end_date"] = pd.to_datetime(engagements_df["end_date"], format="mixed")

print(engagements_df[["start_date", "end_date"]].dtypes)

start_date    datetime64[ns]
end_date      datetime64[ns]
dtype: object


## verify the type changed to datetime64.

In [15]:
print(engagements_df["start_date"].head(10))
print("-" * 25)
print(engagements_df["end_date"].head(10))

0   2023-11-24
1   2025-05-26
2   2024-06-20
3   2023-01-22
4   2025-05-23
5   2023-01-26
6   2023-04-24
7   2023-11-06
8   2025-05-25
9   2025-10-07
Name: start_date, dtype: datetime64[ns]
-------------------------
0   2023-07-02
1   2026-03-23
2   2025-12-15
3   2024-02-28
4   2026-10-21
5   2023-02-07
6   2023-12-19
7   2023-02-13
8   2025-05-14
9   2025-03-18
Name: end_date, dtype: datetime64[ns]


### Use str.replace() remove $ and commas and astype() to convert from string object to valid float64 format

In [16]:
engagements_df["project_fee"] = engagements_df["project_fee"].str.replace("$", "", regex=False)
engagements_df["project_fee"] = engagements_df["project_fee"].str.replace(",", "", regex=False)
engagements_df["project_fee"] = engagements_df["project_fee"].astype(float)
engagements_df["project_fee"] = engagements_df["project_fee"].round(1)

print(engagements_df["project_fee"].dtype)
print("-" * 25)
print(engagements_df["project_fee"])

float64
-------------------------
0      45000.0
1      68000.0
2     230000.0
3     150000.0
4     110000.0
        ...   
85    140000.0
86     75000.0
87     68000.0
88     85000.0
89    110000.0
Name: project_fee, Length: 86, dtype: float64


## Final Check

In [17]:
print("***** Final verification for engagements_raw.csv *****")
print("-" * 55)
print(f"{'Shape':20s}: {engagements_df.shape}")
print("-" * 55)
print(f"{'Duplicates':20s}: {engagements_df.duplicated().sum()}")
print("-" * 55)
print(f"{'Missing values':20s}:")
print(engagements_df.isna().sum())
print("-" * 55)
print(f"{'Dtypes':20s}:")
print(engagements_df.dtypes)

***** Final verification for engagements_raw.csv *****
-------------------------------------------------------
Shape               : (86, 9)
-------------------------------------------------------
Duplicates          : 0
-------------------------------------------------------
Missing values      :
engagement_id        0
client_name          0
practice             0
lead_consultant      0
start_date           0
end_date             0
project_fee          0
status               0
secondary_contact    0
dtype: int64
-------------------------------------------------------
Dtypes              :
engagement_id                object
client_name                  object
practice                     object
lead_consultant              object
start_date           datetime64[ns]
end_date             datetime64[ns]
project_fee                 float64
status                       object
secondary_contact            object
dtype: object


## Save the cleaned dataframe engagements_df to Clean_Data/engagements_clean.csv file

In [18]:
# Cretae Clean_Data folder and save the cleaned dataframe to engagements_clean.csv file
output_dir = "Clean_Data"

try:
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, "engagements_clean.csv")
    engagements_df.to_csv(output_path, index=False)
    print(f"Saved {output_path}")
except FileNotFoundError as e:
    print(f"Invalid file path: {e}")
except Exception as e:
    print(f"Failed to save CSV: {e}")

Saved Clean_Data/engagements_clean.csv


## B. Data Cleaning : timesheets_raw.csv file 

In [19]:
# Define the folder and file name
folder_path = "Raw_Data"
file_name = "timesheets_raw.csv"

# Join folder and file name into
file_path = os.path.join(folder_path, file_name)

print("File to be loaded -->", file_path)
print("-" * 50)
try:
    # Check if the parent folder exists and raise error if the Raw_Data folder is missing 
    if not os.path.isdir(os.path.dirname(file_path) or "."):
        raise FileNotFoundError(f"Folder not found: {os.path.dirname(file_path)}")
    # Check if the specific file exists inside that folder and raise error if the Raw_Data/engagements_raw.csv file is missing 
    if not os.path.isfile(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    # If both checks pass, load the CSV into a engagements_df DataFrame
    timesheets_df = pd.read_csv(file_path)

# Catch the two custom FileNotFoundError cases raised above (missing folder or file)
except FileNotFoundError as e:
    print(f"Error: {e}")
    
# Catch any other unexpected error during the read
except Exception as e:
        print(f"Unexpected error while reading '{file_path}': {e}")

if timesheets_df is not None :
    # Print shape, every column data type, numeric statistics, top 7 rows of data
    print("\n")
    print(f"{file_path} loaded successfully!")
    print()
    print(f"Size of timesheets DataFrame: {timesheets_df.shape[0]} rows, {timesheets_df.shape[1]} columns \n")
    print()
    print(f"dtypes : ")
    print(timesheets_df.dtypes)
    print()
    print(f"Numeric statistics : ")
    display(timesheets_df.describe())
    print()
    display(timesheets_df.head(7))

File to be loaded --> Raw_Data/timesheets_raw.csv
--------------------------------------------------


Raw_Data/timesheets_raw.csv loaded successfully!

Size of timesheets DataFrame: 60 rows, 7 columns 


dtypes : 
entry_id            object
consultant_name     object
engagement_id       object
week_ending         object
hours_logged       float64
billable            object
notes               object
dtype: object

Numeric statistics : 


,hours_logged
count,60.000000
mean,25.703333
std,13.381811
min,-5.500000
25%,15.800000
50%,26.400000
75%,37.700000
max,44.500000


,entry_id,consultant_name,engagement_id,week_ending,hours_logged,billable,notes
0,TS-001,Tomas Herrera,ENG-004,2024-11-08,34.1,No,Risk assessment
1,TS-002,Tomas Herrera,ENG-038,2025-01-24,15.9,No,NaN
2,TS-003,Marcus Webb,ENG-076,2024-11-01,31.0,Y,Documentation
3,TS-004,Tomas Herrera,ENG-034,2024-11-22,34.5,N,Change management planning
4,TS-005,Tomas Herrera,ENG-073,2024-12-20,24.7,Y,Requirements gathering
5,TS-006,Anika Patel,ENG-001,2024-11-22,15.0,No,Weekly status update
6,TS-007,David Chen,NaN,2024-12-20,34.1,Yes,Weekly status update


### B1. Using duplicated() to find repeating rows in timesheets_df

In [20]:
# Get the total number of duplicate rows
print("Total count of duplicate rows in timesheets : ",timesheets_df.duplicated().sum())

# Display all the duplicate rows including the originals 
timesheets_duplicate_rows = timesheets_df[timesheets_df.duplicated(keep=False)] 
timesheets_duplicate_rows

Total count of duplicate rows in timesheets :  4


,entry_id,consultant_name,engagement_id,week_ending,hours_logged,billable,notes
8,TS-009,Tomas Herrera,NaN,2024-11-01,21.8,N,Executive presentation prep
15,TS-009,Tomas Herrera,NaN,2024-11-01,21.8,N,Executive presentation prep
23,TS-023,Rachel Kim,ENG-035,2024-11-29,33.1,No,Weekly status update
30,TS-023,Rachel Kim,ENG-035,2024-11-29,33.1,No,Weekly status update
39,TS-038,Priya Nambiar,NaN,2024-11-22,37.9,N,NaN
45,TS-038,Priya Nambiar,NaN,2024-11-22,37.9,N,NaN
53,TS-051,Tomas Herrera,ENG-067,2024-11-29,15.8,N,Executive presentation prep
55,TS-051,Tomas Herrera,ENG-067,2024-11-29,15.8,N,Executive presentation prep


Rows with entry_id TS-009, TS-023, TS-038, TS-051 are duplicates.

### drop_duplicates() method to remove duplicate rows
Since we know number of duplicates = 4 & total rows = 60, we expect 60-4 = 86 rows after cleaning.

In [21]:
print(f"Number of rows before removing duplictes in timesheets: {len(timesheets_df)}")
timesheets_df = timesheets_df.drop_duplicates()
print(f"Number of rows after removing duplictes in timesheets:  {len(timesheets_df)}")

# Get the total number of duplicate rows - should return 0 now
print("Total count of duplicate rows in timesheets :",timesheets_df.duplicated().sum())

Number of rows before removing duplictes in timesheets: 60
Number of rows after removing duplictes in timesheets:  56
Total count of duplicate rows in timesheets : 0


### B2. Using info() to verify column names, data types, non-null counts¶

In [22]:
timesheets_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 56 entries, 0 to 59
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   entry_id         56 non-null     object 
 1   consultant_name  56 non-null     object 
 2   engagement_id    47 non-null     object 
 3   week_ending      56 non-null     object 
 4   hours_logged     56 non-null     float64
 5   billable         56 non-null     object 
 6   notes            50 non-null     object 
dtypes: float64(1), object(6)
memory usage: 3.5+ KB


### Using isna().sum() to fetch the exact count of missing values in each column of DataFrame.

In [23]:
timesheets_df.isna().sum()

entry_id           0
consultant_name    0
engagement_id      9
week_ending        0
hours_logged       0
billable           0
notes              6
dtype: int64

We see that column - 'engagement_id' has 9 and 'notes' has 6 missing values.

### isna().any(axis=1) finds rows with missing data (NaNs).

In [24]:
timesheets_missing_rows = timesheets_df[timesheets_df.isna().any(axis=1)]
timesheets_missing_rows

,entry_id,consultant_name,engagement_id,week_ending,hours_logged,billable,notes
1,TS-002,Tomas Herrera,ENG-038,2025-01-24,15.9,No,NaN
6,TS-007,David Chen,NaN,2024-12-20,34.1,Yes,Weekly status update
8,TS-009,Tomas Herrera,NaN,2024-11-01,21.8,N,Executive presentation prep
12,TS-013,Rachel Kim,NaN,2025-01-17,6.9,N,Draft deliverable review
26,TS-026,Tomas Herrera,NaN,2024-11-29,9.8,No,Executive presentation prep
34,TS-033,Rachel Kim,NaN,2025-02-07,24.2,N,Proposal development
39,TS-038,Priya Nambiar,NaN,2024-11-22,37.9,N,NaN
40,TS-039,Tomas Herrera,ENG-037,2024-11-08,4.1,Y,NaN
41,TS-040,Lena Kowalski,ENG-061,2025-01-10,25.1,N,NaN
44,TS-043,Tomas Herrera,NaN,2024-12-13,44.2,Y,Client kickoff meeting


In [25]:
# Get the total number of rows with missing values
print(f'There are total {timesheets_missing_rows.shape[0]} rows with missing values in timesheets.')

There are total 14 rows with missing values in timesheets.


### fillna() to fill missing values of 'engagement_id' column with "Unknown" and  'notes' column with "None" :

In [26]:
timesheets_df["engagement_id"] = timesheets_df["engagement_id"].fillna("Unknown")
timesheets_df["notes"] = timesheets_df["notes"].fillna("None")
# isna().sum() to verify if the number for filled column has dropped to 0.
timesheets_df.isna().sum()

entry_id           0
consultant_name    0
engagement_id      0
week_ending        0
hours_logged       0
billable           0
notes              0
dtype: int64

### Apply str.strip() on all string type columns to remove leading/trailing whitespaces

In [27]:
timesheets_df["consultant_name"] = timesheets_df["consultant_name"].str.strip()
timesheets_df["entry_id"] = timesheets_df["entry_id"].str.strip()
timesheets_df["engagement_id"] = timesheets_df["engagement_id"].str.strip()
timesheets_df["billable"] = timesheets_df["billable"].str.strip()
timesheets_df["notes"] = timesheets_df["notes"].str.strip()

# Verify no duplicates rows after trimming whitespaces.
print("Total count of duplicate rows in timesheets_df  :",timesheets_df .duplicated().sum())

Total count of duplicate rows in timesheets_df  : 0


### dtypes to identify Data types of all columns :

In [28]:
timesheets_df.dtypes

entry_id            object
consultant_name     object
engagement_id       object
week_ending         object
hours_logged       float64
billable            object
notes               object
dtype: object

### Apply to_datetime() on week_ending column to convert into valid datetime format

In [29]:
timesheets_df["week_ending"] = pd.to_datetime(timesheets_df["week_ending"], format="mixed")
print(timesheets_df["week_ending"].dtype)

datetime64[ns]


### Data coreection for 'billable' column - Keep values consistent N --> No , Y --> Yes

In [30]:
print(timesheets_df['billable'].head(15).to_list())
print()
print(timesheets_df['billable'].unique())
print()
timesheets_df["billable"].value_counts()

['No', 'No', 'Y', 'N', 'Y', 'No', 'Yes', 'N', 'N', 'Y', 'Y', 'N', 'N', 'N', 'Yes']

['No' 'Y' 'N' 'Yes']



billable
N      25
Y      16
Yes     8
No      7
Name: count, dtype: int64

In [31]:
timesheets_df['billable'] = timesheets_df['billable'].map({'No':'No','N': 'No', 'Yes':'Yes', 'Y': 'Yes'})
print(timesheets_df['billable'].head(15).to_list())
print()
print(timesheets_df['billable'].unique())
print()
timesheets_df["billable"].value_counts()
timesheets_df["billable"].value_counts()

['No', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'No', 'Yes', 'Yes', 'No', 'No', 'No', 'Yes']

['No' 'Yes']



billable
No     32
Yes    24
Name: count, dtype: int64

In [32]:
print("***** Final verification for timesheets_raw.csv *****")
print("-" * 55)
print(f"{'Shape':20s}: {timesheets_df.shape}")
print("-" * 55)
print(f"{'Duplicates':20s}: {timesheets_df.duplicated().sum()}")
print("-" * 55)
print(f"{'Missing values':20s}:")
print(timesheets_df.isna().sum())
print("-" * 55)
print(f"{'Dtypes':20s}:")
print(timesheets_df.dtypes)

***** Final verification for timesheets_raw.csv *****
-------------------------------------------------------
Shape               : (56, 7)
-------------------------------------------------------
Duplicates          : 0
-------------------------------------------------------
Missing values      :
entry_id           0
consultant_name    0
engagement_id      0
week_ending        0
hours_logged       0
billable           0
notes              0
dtype: int64
-------------------------------------------------------
Dtypes              :
entry_id                   object
consultant_name            object
engagement_id              object
week_ending        datetime64[ns]
hours_logged              float64
billable                   object
notes                      object
dtype: object


## Save the cleaned dataframe timesheets_df to Clean_Data/timesheets_clean.csv file

In [33]:
# Save the cleaned dataframe to Clean_Data/timesheets_clean.csv file
output_dir = "Clean_Data"

try:
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, "timesheets_clean.csv")
    timesheets_df.to_csv(output_path, index=False)
    print(f"Saved {output_path}")
except FileNotFoundError as e:
    print(f"Invalid file path: {e}")
except Exception as e:
    print(f"Failed to save CSV: {e}")

Saved Clean_Data/timesheets_clean.csv


## C. Data Cleaning : consultants_raw.csv file 

In [34]:
# Define the folder and file name
folder_path = "Raw_Data"
file_name = "consultants_raw.csv"

# Join folder and file name into
file_path = os.path.join(folder_path, file_name)

print("File to be loaded -->", file_path)
print("-" * 50)
try:
    # Check if the parent folder exists and raise error if the Raw_Data folder is missing 
    if not os.path.isdir(os.path.dirname(file_path) or "."):
        raise FileNotFoundError(f"Folder not found: {os.path.dirname(file_path)}")
    # Check if the specific file exists inside that folder and raise error if the Raw_Data/engagements_raw.csv file is missing 
    if not os.path.isfile(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    # If both checks pass, load the CSV into a engagements_df DataFrame
    consultants_df = pd.read_csv(file_path)

# Catch the two custom FileNotFoundError cases raised above (missing folder or file)
except FileNotFoundError as e:
    print(f"Error: {e}")
    
# Catch any other unexpected error during the read
except Exception as e:
        print(f"Unexpected error while reading '{file_path}': {e}")

if timesheets_df is not None :
    # Print shape, every column data type, numeric statistics, top 7 rows of data
    print("\n")
    print(f"{file_path} loaded successfully!")
    print()
    print(f"Size of consultants DataFrame: {consultants_df.shape[0]} rows, {consultants_df.shape[1]} columns \n")
    print()
    print(f"dtypes : ")
    print(consultants_df.dtypes)
    print()
    print(f"Numeric statistics : ")
    display(consultants_df.describe())
    print()
    display(consultants_df.head(7))

File to be loaded --> Raw_Data/consultants_raw.csv
--------------------------------------------------


Raw_Data/consultants_raw.csv loaded successfully!

Size of consultants DataFrame: 8 rows, 5 columns 


dtypes : 
consultant_name     object
practice            object
level               object
billable_rate      float64
hire_date           object
dtype: object

Numeric statistics : 


,billable_rate
count,8.000000
mean,258.750000
std,89.432736
min,150.000000
25%,215.000000
50%,245.000000
75%,306.250000
max,425.000000


,consultant_name,practice,level,billable_rate,hire_date
0,Priya Nambiar,Strategy,Partner,425.0,2015-03-15
1,Marcus Webb,Digital,Lead,310.0,2017-06-01
2,Jordan Reeves,Operations,Senior,235.0,2019-01-10
3,Anika Patel,People & Change,Senior,240.0,2018-09-22
4,David Chen,Digital,Junior,155.0,2022-07-11
5,Rachel Kim,Strategy,Senior,250.0,2020-04-01
6,Tomás Herrera,Operations,Lead,305.0,2017-11-15


### C1. Using duplicated() to find repeating rows in consultants_df

In [35]:
# Get the total number of duplicate rows
print("Total count of duplicate rows in consultants : ",consultants_df.duplicated().sum())

# Display all the duplicate rows including the originals 
consultants_duplicate_rows = consultants_df[consultants_df.duplicated(keep=False)] 
consultants_duplicate_rows

Total count of duplicate rows in consultants :  0


,consultant_name,practice,level,billable_rate,hire_date


There are no duplicate rows.

### C2. Using info() to verify column names, data types, non-null counts¶

In [36]:
consultants_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   consultant_name  8 non-null      object 
 1   practice         8 non-null      object 
 2   level            8 non-null      object 
 3   billable_rate    8 non-null      float64
 4   hire_date        8 non-null      object 
dtypes: float64(1), object(4)
memory usage: 448.0+ bytes


In [37]:
# See how many missing data points we have per column
consultants_df.isna().sum()

consultant_name    0
practice           0
level              0
billable_rate      0
hire_date          0
dtype: int64

There are no mising values.

### Apply str.strip() on all string type columns to remove leading/trailing whitespaces

In [38]:
consultants_df["consultant_name"] = consultants_df["consultant_name"].str.strip()
consultants_df["practice"] = consultants_df["practice"].str.strip()
consultants_df["level"] = consultants_df["level"].str.strip()

# Verify no duplicates rows after trimming whitespaces.
print("Total count of duplicate rows in consultants_df :",consultants_df.duplicated().sum())

Total count of duplicate rows in consultants_df : 0


### dtypes to identify Data types of all columns :¶

In [39]:
consultants_df.dtypes

consultant_name     object
practice            object
level               object
billable_rate      float64
hire_date           object
dtype: object

### Apply to_datetime() on 'hire_date' column to convert into valid datetime format

In [40]:
consultants_df["hire_date"] = pd.to_datetime(consultants_df["hire_date"], format="mixed")

print(consultants_df["hire_date"].dtype)
consultants_df.head()

datetime64[ns]


,consultant_name,practice,level,billable_rate,hire_date
0,Priya Nambiar,Strategy,Partner,425.0,2015-03-15
1,Marcus Webb,Digital,Lead,310.0,2017-06-01
2,Jordan Reeves,Operations,Senior,235.0,2019-01-10
3,Anika Patel,People & Change,Senior,240.0,2018-09-22
4,David Chen,Digital,Junior,155.0,2022-07-11


### Final Check

In [41]:
print("***** Final verification for consultants_raw.csv *****")
print("-" * 55)
print(f"{'Shape':20s}: {consultants_df.shape}")
print("-" * 55)
print(f"{'Duplicates':20s}: {consultants_df.duplicated().sum()}")
print("-" * 55)
print(f"{'Missing values':20s}:")
print(consultants_df.isna().sum())
print("-" * 55)
print(f"{'Dtypes':20s}:")
print(consultants_df.dtypes)

***** Final verification for consultants_raw.csv *****
-------------------------------------------------------
Shape               : (8, 5)
-------------------------------------------------------
Duplicates          : 0
-------------------------------------------------------
Missing values      :
consultant_name    0
practice           0
level              0
billable_rate      0
hire_date          0
dtype: int64
-------------------------------------------------------
Dtypes              :
consultant_name            object
practice                   object
level                      object
billable_rate             float64
hire_date          datetime64[ns]
dtype: object


## Save the cleaned dataframe consultants_df to Clean_Data/consultants_clean.csv file

In [42]:
# Save the cleaned dataframe to Clean_Data/timesheets_clean.csv file
output_dir = "Clean_Data"

try:
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, "consultants_clean.csv")
    consultants_df.to_csv(output_path, index=False)
    print(f"Saved {output_path}")
except FileNotFoundError as e:
    print(f"Invalid file path: {e}")
except Exception as e:
    print(f"Failed to save CSV: {e}")

Saved Clean_Data/consultants_clean.csv


### Merge timesheets_clean.csv, engagements_clean.csv & consultants_clean.csv to meridian_merged.csv 

In [43]:
import pandas as pd
from pathlib import Path

Clean_Data = Path("Clean_Data")

files = {
    "timesheets": Clean_Data / "timesheets_clean.csv",
    "engagements": Clean_Data / "engagements_clean.csv",
    "consultants": Clean_Data / "consultants_clean.csv",
}

try:
    if not Clean_Data.exists():
        raise FileNotFoundError(f"Folder not found: {Clean_Data.resolve()}")
    if not Clean_Data.is_dir():
        raise NotADirectoryError(f"Path exists but is not a folder: {Clean_Data.resolve()}")

    for name, path in files.items():
        if not path.exists():
            raise FileNotFoundError(f"Missing file for '{name}': {path.resolve()}")

    timesheets = pd.read_csv(files["timesheets"])
    print("timesheets.csv loaded successfully.")
    print("-" * 35)
    print(f"timesheets: {timesheets.shape[0]:,} rows, {timesheets.shape[1]:,} columns")
    display(timesheets.head())

    engagements = pd.read_csv(files["engagements"])
    print("engagements.csv loaded successfully.")
    print("-" * 35)
    print(f"engagements: {engagements.shape[0]:,} rows, {engagements.shape[1]:,} columns")
    display(engagements.head())

    consultants = pd.read_csv(files["consultants"])
    print("consultants.csv loaded successfully.")
    print("-" * 35)
    print(f"consultants: {consultants.shape[0]:,} rows, {consultants.shape[1]:,} columns")
    display(consultants.head())

    print("All files loaded successfully.")

except FileNotFoundError as e:
    print(f"File/Folder Error: {e}")
except Exception as e:
    print(f"Unexpected Error ({type(e).__name__}): {e}")

timesheets.csv loaded successfully.
-----------------------------------
timesheets: 56 rows, 7 columns


,entry_id,consultant_name,engagement_id,week_ending,hours_logged,billable,notes
0,TS-001,Tomas Herrera,ENG-004,2024-11-08,34.1,No,Risk assessment
1,TS-002,Tomas Herrera,ENG-038,2025-01-24,15.9,No,NaN
2,TS-003,Marcus Webb,ENG-076,2024-11-01,31.0,Yes,Documentation
3,TS-004,Tomas Herrera,ENG-034,2024-11-22,34.5,No,Change management planning
4,TS-005,Tomas Herrera,ENG-073,2024-12-20,24.7,Yes,Requirements gathering


engagements.csv loaded successfully.
-----------------------------------
engagements: 86 rows, 9 columns


,engagement_id,client_name,practice,lead_consultant,start_date,end_date,project_fee,status,secondary_contact
0,ENG-001,Apex Financial,Digital,Jordan Reeves,2023-11-24,2023-07-02,45000.0,Active,Not assigned
1,ENG-002,Orion Energy,People & Change,Anika Patel,2025-05-26,2026-03-23,68000.0,On Hold,Not assigned
2,ENG-003,Vanguard Logistics,Strategy,Tomas Herrera,2024-06-20,2025-12-15,230000.0,Active,Hannah Firth
3,ENG-004,Apex Financial,Operations,Anika Patel,2023-01-22,2024-02-28,150000.0,Active,Not assigned
4,ENG-005,Titan Manufacturing,Operations,Rachel Kim,2025-05-23,2026-10-21,110000.0,Completed,Not assigned


consultants.csv loaded successfully.
-----------------------------------
consultants: 8 rows, 5 columns


,consultant_name,practice,level,billable_rate,hire_date
0,Priya Nambiar,Strategy,Partner,425.0,2015-03-15
1,Marcus Webb,Digital,Lead,310.0,2017-06-01
2,Jordan Reeves,Operations,Senior,235.0,2019-01-10
3,Anika Patel,People & Change,Senior,240.0,2018-09-22
4,David Chen,Digital,Junior,155.0,2022-07-11


All files loaded successfully.


In [44]:
# Identify shared keys
print("timesheets columns: ", list(timesheets.columns))
print()
print("engagements columns:", list(engagements.columns))
print()
print("consultants columns: ", list(consultants.columns))

timesheets columns:  ['entry_id', 'consultant_name', 'engagement_id', 'week_ending', 'hours_logged', 'billable', 'notes']

engagements columns: ['engagement_id', 'client_name', 'practice', 'lead_consultant', 'start_date', 'end_date', 'project_fee', 'status', 'secondary_contact']

consultants columns:  ['consultant_name', 'practice', 'level', 'billable_rate', 'hire_date']


### Common keys
timesheets and engagements have 'engagement_id' in common 

timesheets and consultants have 'consultant_name' in common

### Before merge, compare the unique values in each key column and verify key mismatches using set()

In [45]:
ts_consultants = set(timesheets["consultant_name"].dropna().unique())
csv_consultants = set(consultants["consultant_name"].dropna().unique())

print("In timesheets but NOT in consultants:")
print(ts_consultants - csv_consultants)

print("\nIn consultants but NOT in timesheets:")
print(csv_consultants - ts_consultants)

In timesheets but NOT in consultants:
{'David  Chen', 'Lena  Kowalski', 'Tomas Herrera'}

In consultants but NOT in timesheets:
{'Tomás Herrera'}


### Since 'consultant_name' column has inconsistent  values, fix and then merge :

'Tomas Herrera'  Vs 'Tomás Herrera' ---> 'a' and 'á'

'Lena  Kowalski' Vs 'Lena Kowalski' --->  white space mismatch

'David  Chen'   Vs  'David Chen'    --->  white space mismatch

In [46]:
# Set consistent values across timesheets and consultants
import re, unicodedata

def normalize_name(name):
    if pd.isna(name):
        return name
    name = unicodedata.normalize('NFKD', str(name))
    name = ''.join(c for c in name if not unicodedata.combining(c))   # strip accents
    #name = re.sub(r'\s+', ' ', name).strip().lower()                 # collapse spaces, lowercase
    name = re.sub(r'\s+', ' ', name).strip()                          # collapse spaces
    return name

for df in (timesheets, consultants):
    df['consultant_name'] = df['consultant_name'].apply(normalize_name)

In [47]:
# Verify the set output - NULL set in case of matching column values
ts_consultants = set(timesheets["consultant_name"].dropna().unique())
csv_consultants = set(consultants["consultant_name"].dropna().unique())

print("In timesheets but NOT in consultants:")
print(ts_consultants - csv_consultants)

print("\nIn consultants but NOT in timesheets:")
print(csv_consultants - ts_consultants)


In timesheets but NOT in consultants:
set()

In consultants but NOT in timesheets:
set()


#### Merge timesheets, engagements & consultants together in a single dataframe  'timesheets_engagements_consultants_merge'
Use **left joins** at each step such that crucial timesheet data survives. Missing engagement or consultant records will show up as `NaN`

### Step 1: Merge timesheets + engagements on 'engagement_id' column:

In [48]:
timesheets_engagements_merge = pd.merge(
    timesheets,
    engagements,
    on="engagement_id",
    how="left"
)

print(f"Merge Step 1: timesheets and engagements merged to final shape {timesheets_engagements_merge.shape}")
timesheets_engagements_merge.head(3)

Merge Step 1: timesheets and engagements merged to final shape (56, 15)


,entry_id,consultant_name,engagement_id,week_ending,hours_logged,billable,notes,client_name,practice,lead_consultant,start_date,end_date,project_fee,status,secondary_contact
0,TS-001,Tomas Herrera,ENG-004,2024-11-08,34.1,No,Risk assessment,Apex Financial,Operations,Anika Patel,2023-01-22,2024-02-28,150000.0,Active,Not assigned
1,TS-002,Tomas Herrera,ENG-038,2025-01-24,15.9,No,NaN,Orion Energy,Digital,Lena Kowalski,2024-07-16,2024-11-22,55000.0,Completed,Camille Rousseau
2,TS-003,Marcus Webb,ENG-076,2024-11-01,31.0,Yes,Documentation,Orion Energy,Strategy,Rachel Kim,2024-08-08,2024-11-12,230000.0,On Hold,Not assigned


### Step 2: Merge step1 result + consultants on 'consultant_name' column:

In [49]:
timesheets_engagements_consultants_merge = pd.merge(
    timesheets_engagements_merge,
    consultants,
    on="consultant_name",
    how="left",
    suffixes=("_engagement", "_consultant")
)

print(f"Merge Step 2: timesheets_engagements_merge and consultants merged to final shape {timesheets_engagements_consultants_merge.shape}")
timesheets_engagements_consultants_merge.head(3)

Merge Step 2: timesheets_engagements_merge and consultants merged to final shape (56, 19)


,entry_id,consultant_name,engagement_id,week_ending,hours_logged,billable,notes,client_name,practice_engagement,lead_consultant,start_date,end_date,project_fee,status,secondary_contact,practice_consultant,level,billable_rate,hire_date
0,TS-001,Tomas Herrera,ENG-004,2024-11-08,34.1,No,Risk assessment,Apex Financial,Operations,Anika Patel,2023-01-22,2024-02-28,150000.0,Active,Not assigned,Operations,Lead,305.0,2017-11-15
1,TS-002,Tomas Herrera,ENG-038,2025-01-24,15.9,No,NaN,Orion Energy,Digital,Lena Kowalski,2024-07-16,2024-11-22,55000.0,Completed,Camille Rousseau,Operations,Lead,305.0,2017-11-15
2,TS-003,Marcus Webb,ENG-076,2024-11-01,31.0,Yes,Documentation,Orion Energy,Strategy,Rachel Kim,2024-08-08,2024-11-12,230000.0,On Hold,Not assigned,Digital,Lead,310.0,2017-06-01


In [50]:
print(f"Shape: {timesheets_engagements_consultants_merge.shape}")
print("Columns:")
for column in timesheets_engagements_consultants_merge.columns:
    print(f"- {column}")

Shape: (56, 19)
Columns:
- entry_id
- consultant_name
- engagement_id
- week_ending
- hours_logged
- billable
- notes
- client_name
- practice_engagement
- lead_consultant
- start_date
- end_date
- project_fee
- status
- secondary_contact
- practice_consultant
- level
- billable_rate
- hire_date


In [51]:
timesheets_engagements_consultants_merge.dtypes

entry_id                object
consultant_name         object
engagement_id           object
week_ending             object
hours_logged           float64
billable                object
notes                   object
client_name             object
practice_engagement     object
lead_consultant         object
start_date              object
end_date                object
project_fee            float64
status                  object
secondary_contact       object
practice_consultant     object
level                   object
billable_rate          float64
hire_date               object
dtype: object

## Validate the merged result for row count, missing values for timesheets dataset

In [52]:
print(f"Original timesheets : {len(timesheets):,} rows")
print(f"After full merge    : {len(timesheets_engagements_consultants_merge):,} rows")
print(f"Rows gained/lost    : {len(timesheets_engagements_consultants_merge) - len(timesheets):,}")
print("-" * 50)
merged_null_counts = timesheets_engagements_consultants_merge.isnull().sum()
print("Null counts :", merged_null_counts)

Original timesheets : 56 rows
After full merge    : 56 rows
Rows gained/lost    : 0
--------------------------------------------------
Null counts : entry_id               0
consultant_name        0
engagement_id          0
week_ending            0
hours_logged           0
billable               0
notes                  6
client_name            9
practice_engagement    9
lead_consultant        9
start_date             9
end_date               9
project_fee            9
status                 9
secondary_contact      9
practice_consultant    0
level                  0
billable_rate          0
hire_date              0
dtype: int64


Since we see there are many rows with NULL entries, these are because of engagement_id is 'Unknown'. 
Since Unknown  engagement_id cannot be mapped to any client_name, we cannot dervie any meaningful KPIs out of this data. 
Hence we are dropping these entries with engagement_id = 'Unknown'. 

In [53]:
print("Merged data size before:", timesheets_engagements_consultants_merge.shape)

timesheets_engagements_consultants_merge = timesheets_engagements_consultants_merge[
    timesheets_engagements_consultants_merge['engagement_id'] != 'Unknown']

print("Merged data size before:", timesheets_engagements_consultants_merge.shape)

Merged data size before: (56, 19)
Merged data size before: (47, 19)


In [54]:
# Filling 'notes' column missing values with 'None'
timesheets_engagements_consultants_merge["notes"] = timesheets_engagements_consultants_merge["notes"].fillna("Unknown entry")
merged_null_counts = timesheets_engagements_consultants_merge.isnull().sum()
print("Null counts :", merged_null_counts)

Null counts : entry_id               0
consultant_name        0
engagement_id          0
week_ending            0
hours_logged           0
billable               0
notes                  0
client_name            0
practice_engagement    0
lead_consultant        0
start_date             0
end_date               0
project_fee            0
status                 0
secondary_contact      0
practice_consultant    0
level                  0
billable_rate          0
hire_date              0
dtype: int64


In [55]:
display(timesheets_engagements_consultants_merge.head())

,entry_id,consultant_name,engagement_id,week_ending,hours_logged,billable,notes,client_name,practice_engagement,lead_consultant,start_date,end_date,project_fee,status,secondary_contact,practice_consultant,level,billable_rate,hire_date
0,TS-001,Tomas Herrera,ENG-004,2024-11-08,34.1,No,Risk assessment,Apex Financial,Operations,Anika Patel,2023-01-22,2024-02-28,150000.0,Active,Not assigned,Operations,Lead,305.0,2017-11-15
1,TS-002,Tomas Herrera,ENG-038,2025-01-24,15.9,No,Unknown entry,Orion Energy,Digital,Lena Kowalski,2024-07-16,2024-11-22,55000.0,Completed,Camille Rousseau,Operations,Lead,305.0,2017-11-15
2,TS-003,Marcus Webb,ENG-076,2024-11-01,31.0,Yes,Documentation,Orion Energy,Strategy,Rachel Kim,2024-08-08,2024-11-12,230000.0,On Hold,Not assigned,Digital,Lead,310.0,2017-06-01
3,TS-004,Tomas Herrera,ENG-034,2024-11-22,34.5,No,Change management planning,Apex Financial,People & Change,Priya Nambiar,2023-02-15,2024-11-17,230000.0,On Hold,Raj Anand,Operations,Lead,305.0,2017-11-15
4,TS-005,Tomas Herrera,ENG-073,2024-12-20,24.7,Yes,Requirements gathering,Titan Manufacturing,People & Change,Rachel Kim,2025-11-09,2025-11-12,85000.0,Completed,Not assigned,Operations,Lead,305.0,2017-11-15


## Save the final merged file as 'clean_merged.csv' to Clean_Data Folder:

In [56]:
# Save the timesheets_engagements_consultants_merge dataframe to Clean_Data/clean_merged.csv file
output_dir = "Clean_Data"

try:
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, "clean_merged.csv")
    timesheets_engagements_consultants_merge.to_csv(output_path, index=False)
    print(f"Saved {output_path}: {timesheets_engagements_consultants_merge.shape[0]:,} rows, "
          f"{timesheets_engagements_consultants_merge.shape[1]:,} columns")
except FileNotFoundError as e:
    print(f"Invalid file path: {e}")
except Exception as e:
    print(f"Failed to save CSV: {e}")

Saved Clean_Data/clean_merged.csv: 47 rows, 19 columns
